In [1]:
pip install faker pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\agusb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import random
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from faker import Faker

In [3]:
fake = Faker('es_ES')
Faker.seed(
    42
)
random.seed(42)
np.random.seed(42)

# Tabla vehicles

In [4]:
# Definimos las reglas de negocio lógicas para los vehículos de FleetLogix
vehicle_configs = [
    {
        'type': 'Camión Grande',
        'min_cap': 8000.0,
        'max_cap': 15000.0,
        'fuels': ['Diesel'],
    },
    {
        'type': 'Camión Mediano',
        'min_cap': 3500.0,
        'max_cap': 7999.0,
        'fuels': ['Diesel', 'Gasolina'],
    },
    {
        'type': 'Van',
        'min_cap': 800.0,
        'max_cap': 3499.0,
        'fuels': ['Diesel', 'Gasolina', 'Eléctrico'],
    },
    {
        'type': 'Motocicleta',
        'min_cap': 50.0,
        'max_cap': 250.0,
        'fuels': ['Gasolina', 'Eléctrico'],
    },
]

vehicles = []
used_plates = set()

In [5]:
# Función para asegurar que las patentes sean únicas
def generate_unique_plate():
    while True:
        # fake.bothify mezcla letras (?) y números (#) al azar
        plate = fake.bothify(text='??###??').upper()
        if plate not in used_plates:
            used_plates.add(plate)
            return plate

In [6]:
# 3. Bucle para generar exactamente los 200 registros pedidos
for i in range(1, 201):
    config = random.choice(vehicle_configs)
    vehicle_type = config['type']

In [7]:
# Capacidad y combustible coherentes según el tipo de vehículo
capacity_kg = round(
    random.uniform(config['min_cap'], config['max_cap']), 2
)
fuel_type = random.choice(config['fuels'])

In [8]:
# Faker genera una fecha de adquisición realista en los últimos 5 años
acquisition_date = fake.date_between(
        start_date='-5y', end_date='-1y'
    ).strftime('%Y-%m-%d')

In [9]:
# Estados con probabilidades lógicas (la mayoría activos)
status = np.random.choice(
        ['active', 'maintenance', 'inactive'], p=[0.85, 0.10, 0.05]
    )

vehicles.append({
    'vehicle_id': i,
    'license_plate': generate_unique_plate(),
    'vehicle_type': vehicle_type,
    'capacity_kg': capacity_kg,
    'fuel_type': fuel_type,
    'acquisition_date': acquisition_date,
    'status': status,
    })

In [10]:
# 4. Lo convertimos a un DataFrame de pandas y lo guardamos en CSV
df_vehicles = pd.DataFrame(vehicles)
df_vehicles.to_csv('vehicles.csv', index=False)

print('¡Tabla vehicles generada con éxito!')
print(df_vehicles.head(10))

¡Tabla vehicles generada con éxito!
   vehicle_id license_plate   vehicle_type  capacity_kg fuel_type  \
0         200       PO104IV  Camión Grande      8354.12    Diesel   

  acquisition_date  status  
0       2024-07-15  active  


# Tabla Drivers

In [11]:
drivers = []
used_employee_codes = set()
used_license_numbers = set()

In [12]:
# Rango de fechas coherente para las contrataciones (últimos 5 años)
start_hire_date = datetime(2021, 1, 1)
end_hire_date = datetime(2026, 6, 1)

In [13]:
# 2. Bucle para generar exactamente los 400 registros de conductores
for i in range(1, 401):

  # Generar código de empleado único
  while True:
    employee_code = f'EMP-{random.randint(1000, 9999)}'
    if employee_code not in used_employee_codes:
      used_employee_codes.add(employee_code)
      break

  # Generar número de licencia único
  while True:
    license_number = f'LIC-{random.randint(10000000, 99999999)}'
    if license_number not in used_license_numbers:
      used_license_numbers.add(license_number)
      break

In [14]:
#nombre y apellidos con faker
first_name = fake.first_name() 
last_name = fake.last_name()

In [15]:
# Fecha de contratación aleatoria actualizada
hire_date = start_hire_date + timedelta(
      days=random.randint(0, (end_hire_date - start_hire_date).days)
  )

In [16]:
# Consistencia temporal: La licencia expira entre 2 y 5 años DESPUÉS de ser contratado
license_expiry = hire_date + timedelta(days=random.randint(730, 1825))

In [17]:
# Teléfono de contacto simulado
phone = f'+54911{random.randint(10000000, 99999999)}'

In [18]:
# Estado del conductor (la mayoría activos)
status = np.random.choice(['active', 'inactive'], p=[0.90, 0.10])

drivers.append({
      'driver_id': i,
      'employee_code': employee_code,
      'first_name': first_name,
      'last_name': last_name,
      'license_number': license_number,
      'license_expiry': license_expiry.strftime('%Y-%m-%d'),
      'phone': phone,
      'hire_date': hire_date.strftime('%Y-%m-%d'),
      'status': status,
  })

In [19]:
# 3. Convertir a DataFrame y guardar en CSV
df_drivers = pd.DataFrame(drivers)
df_drivers.to_csv('drivers.csv', index=False)

print('¡Tabla drivers generada con éxito y adaptada al 2026!')
print(df_drivers.head(10))

¡Tabla drivers generada con éxito y adaptada al 2026!
   driver_id employee_code first_name last_name license_number license_expiry  \
0        400      EMP-3344   Cristian     Bonet   LIC-41163805     2027-01-04   

            phone   hire_date    status  
0  +5491171682508  2023-08-03  inactive  


# Tabla Routes

In [20]:
# Fijar semilla para reproducibilidad
random.seed(42)
np.random.seed(42)

In [21]:
# 1. Definir las 5 ciudades principales y distancias base reales aproximadas
cities = ['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza', 'San Miguel de Tucumán']

city_distances = {
    ('Buenos Aires', 'Córdoba'): (700, 8.5, 4500),
    ('Buenos Aires', 'Rosario'): (300, 3.5, 2000),
    ('Buenos Aires', 'Mendoza'): (1050, 13.0, 6500),
    ('Buenos Aires', 'San Miguel de Tucumán'): (1250, 15.0, 7500),
    ('Córdoba', 'Rosario'): (400, 4.5, 2500),
    ('Córdoba', 'Mendoza'): (600, 7.5, 3500),
    ('Córdoba', 'San Miguel de Tucumán'): (550, 7.0, 3000),
    ('Rosario', 'Mendoza'): (850, 10.0, 5000),
    ('Rosario', 'San Miguel de Tucumán'): (950, 11.5, 5500),
    ('Mendoza', 'San Miguel de Tucumán'): (980, 12.0, 6000),
}

routes = []
route_id = 1

In [22]:
# 2. Generar combinaciones de rutas entre las ciudades hasta completar 50 registros
for c1 in cities:
  for c2 in cities:
    if c1 != c2 and route_id <= 50:
      pair = (c1, c2) if (c1, c2) in city_distances else (c2, c1)
      base_dist, base_dur, base_toll = city_distances[pair]

In [23]:
# Variación realista del 5% al 10% por ruta
dist = round(base_dist * random.uniform(0.95, 1.05), 2)
dur = round(base_dur * random.uniform(0.95, 1.05), 2)
toll = round(base_toll * random.uniform(0.90, 1.10), 2)

routes.append({
          'route_id': route_id,
          'route_code': f'RTE-{100+route_id}',
          'origin_city': c1,
          'destination_city': c2,
          'distance_km': dist,
          'estimated_duration_hours': dur,
          'toll_cost': toll,
})
route_id += 1

In [24]:
# 3. Convertir a DataFrame y guardar en CSV
df_routes = pd.DataFrame(routes)
df_routes.to_csv('routes.csv', index=False)

print('¡Tabla routes generada con éxito (50 registros)!')
print(df_routes.head(10))

¡Tabla routes generada con éxito (50 registros)!
   route_id route_code            origin_city       destination_city  \
0         1    RTE-101  San Miguel de Tucumán  San Miguel de Tucumán   

   distance_km  estimated_duration_hours  toll_cost  
0       993.66                     11.43    5730.04  
